# depth_bench on Kaggle

Runs the full benchmark (`depth_bench/run_all.py`) over several Kaggle sessions of up to 12 h. Each session continues where the previous one stopped.

**Setup (once)**
1. Upload `D:\datasets\MODEST_colab` as a private Kaggle dataset (see `depth_bench/README.md`).
2. Notebook settings: **Accelerator: GPU T4 x2**, **Internet: on**. Add the dataset as input.

**Each session**
1. From the 2nd session on: add the **output of this notebook's previous version** as an input (Add Input → Your Work → this notebook). The runs so far are copied from it.
2. **Save Version → Save & Run All (Commit)**. It runs without the browser for up to 12 h; the cells below stop all work in time and the results are saved as this version's output.

Each T4 runs its own `run_all` process; they share the jobs through lock files.

In [ ]:
import glob, os, subprocess, sys, time
T0 = time.time()
SESSION_HOURS = 11.5          # Kaggle stops a session at 12 h; everything must be saved before that
TMP = "/kaggle/tmp"           # local disk, not saved: code, packages' caches, extracted data
RUNS = "/kaggle/working/MODEST_runs"   # saved as the version's output (limit 20 GB)
DATASET = [os.path.dirname(p) for p in glob.glob("/kaggle/input/*/manifest.csv") + glob.glob("/kaggle/input/*/*/manifest.csv")][0]
print("dataset:", DATASET)
os.makedirs(TMP, exist_ok=True)

## Code: this repository and the four model repositories (fixed versions)

In [ ]:
%%bash
set -e
cd /kaggle/tmp
if [ -d RealCameraIndependent ]; then git -C RealCameraIndependent pull -q; else git clone -q https://github.com/sleekEagle/RealCameraIndependent; fi
mkdir -p code_ext && cd code_ext
clone() { [ -d "$1" ] || git clone -q "$2" "$1"; git -C "$1" checkout -q "$3"; }
clone Depth-Anything-V2 https://github.com/DepthAnything/Depth-Anything-V2   a561b849ebae10a6f5ef49e26c83cbbcd36c71bf
clone UniDepth          https://github.com/lpiccinelli-eth/UniDepth          8d8cfe4c7ee15297099983607febf0d4f32eb3d6
clone Metric3D          https://github.com/YvanYin/Metric3D                  eb5b6fac0dc155e4e52f576e304fbf11655ff339
clone Depth-Anything-3  https://github.com/ByteDance-Seed/Depth-Anything-3   3d835ec1a5802d64a8b8b15f817a1ab54809bfe4
echo code ready

## Packages (Kaggle's PyTorch is kept; `--no-deps` because UniDepth and DA3 pin conflicting numpy versions)

In [ ]:
%%bash
set -e
pip install -q opencv-python-headless tifffile imagecodecs pillow pandas scipy tqdm huggingface_hub safetensors timm einops addict omegaconf wandb h5py tabulate termcolor imageio mmengine
pip install -q --no-deps -e /kaggle/tmp/code_ext/UniDepth -e /kaggle/tmp/code_ext/Depth-Anything-3
python -c "import torch; print('torch', torch.__version__, 'GPUs', torch.cuda.device_count())"

## Data (extracts the tars to local disk, or links the folders if Kaggle unpacked them)

In [ ]:
os.chdir("/kaggle/tmp/RealCameraIndependent")
!python -m depth_bench.kaggle_data --input {DATASET} --out {TMP}/MODEST_depth

## Runs so far: copied from the previous version's output (if added as input)

In [ ]:
prev = [os.path.dirname(p) for p in glob.glob("/kaggle/input/*/MODEST_runs/eval") + glob.glob("/kaggle/input/*/*/MODEST_runs/eval")]
if prev and not os.path.isdir(RUNS):
    best = max(prev, key=lambda d: len(glob.glob(f"{d}/eval/*/summary.csv")) + len(glob.glob(f"{d}/finetune/*/*.pt")))
    print("restoring", best)
    subprocess.run(["cp", "-r", best, RUNS], check=True)
os.makedirs(RUNS, exist_ok=True)
print(len(glob.glob(f"{RUNS}/eval/*/summary.csv")), "evaluations done so far")

## Run: one `run_all` per GPU until the session time is used up

In [ ]:
os.environ.update({
    "MODEST_DEPTH_ROOT": f"{TMP}/MODEST_depth",
    "MODEST_FOCUS_CSV": glob.glob(f"{DATASET}/**/dfocus_aperture_pairs.csv", recursive=True)[0],
    "DEPTH_BENCH_CODE_EXT": f"{TMP}/code_ext",
    "DEPTH_BENCH_RUNS": RUNS,
    "HF_HOME": f"{TMP}/hf_cache",
    "TORCH_HOME": f"{TMP}/torch_cache",
})
import torch
n_gpu = max(1, torch.cuda.device_count())
hours = SESSION_HOURS - (time.time() - T0) / 3600
workers = max(1, (os.cpu_count() or 2) // n_gpu)
print(f"{n_gpu} GPU(s), {hours:.1f} h left, {workers} data workers per GPU")
procs = []
for g in range(n_gpu):
    cmd = [sys.executable, "-u", "-m", "depth_bench.run_all", "--hours", f"{hours:.2f}", "--workers", str(workers)]
    if g == 0:
        cmd.append("--clear_locks")  # locks left by the previous (killed) session
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(g))
    log = open(f"{RUNS}/run_all_gpu{g}.log", "a")
    procs.append(subprocess.Popen(cmd, env=env, stdout=log, stderr=subprocess.STDOUT))
    time.sleep(20)  # let GPU 0 clear old locks before GPU 1 starts claiming
for p in procs:
    p.wait()
for g in range(n_gpu):
    print(f"===== GPU {g}")
    print("".join(open(f"{RUNS}/run_all_gpu{g}.log").readlines()[-25:]))

## Results so far

In [ ]:
import pandas as pd
rows = []
for f in sorted(glob.glob(f"{RUNS}/eval/*/summary.csv")):
    s = pd.read_csv(f); s.insert(0, "run", f.split("/")[-2]); rows.append(s[s.group.str.startswith("S")])
print(len(rows), "evaluations done;", sorted(set(os.listdir(f"{RUNS}/finetune"))) if os.path.isdir(f"{RUNS}/finetune") else [])
!du -sh {RUNS}
pd.concat(rows)[["run", "group", "value", "n_crops", "absrel", "absrel_med", "delta1", "si_absrel", "scale", "bf1"]] if rows else "no results yet"